# NULL: unknown is not zero

Practice notebook for the datafromzero SQL lesson **NULL: unknown is not zero**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. Find the missing value

In [ ]:
con.sql("""
SELECT * FROM orders WHERE customer_id IS NULL
""").show()

### Step 2. Try `= NULL`

In [ ]:
con.sql("""
SELECT * FROM orders WHERE customer_id = NULL
""").show()

### Step 3. Meet the first trap

In [ ]:
con.sql("""
SELECT COUNT(*) AS orders FROM orders WHERE customer_id <> 7
""").show()

In [ ]:
con.sql("""
SELECT COUNT(*) AS orders FROM orders WHERE customer_id = 7
""").show()

### Step 4. Ask for the unknown on purpose

In [ ]:
con.sql("""
SELECT COUNT(*) AS orders
FROM orders
WHERE customer_id <> 7 OR customer_id IS NULL
""").show()

### Step 5. See that NULL is not 0 and not empty text

In [ ]:
con.sql("""
SELECT 0 IS NULL AS zero_is_null, '' IS NULL AS empty_text_is_null, NULL IS NULL AS null_is_null
""").show()

### Step 6. Meet the second trap: NOT IN

In [ ]:
con.sql("""
SELECT customer_id
FROM customers
WHERE customer_id <= 8 AND customer_id NOT IN (1, 3, 7)
""").show()

In [ ]:
con.sql("""
SELECT customer_id
FROM customers
WHERE customer_id <= 8 AND customer_id NOT IN (1, 3, 7, NULL)
""").show()

### Step 7. Replace NULL with a default

In [ ]:
con.sql("""
SELECT customer_id, referred_by, COALESCE(referred_by, 0) AS referrer_id
FROM customers
WHERE customer_id <= 8
ORDER BY customer_id
""").show()

### Step 8. Verify

In [ ]:
con.sql("""
SELECT COUNT(*) AS orders FROM orders WHERE customer_id IS NOT NULL
""").show()

## Exercise 1. Old Town orders from loyalty customers

Old Town is `store_id` 1. The first three lines are given and count all of Old Town's orders. Add the line that keeps only
the orders whose customer is not missing.

In [ ]:
con.sql("""
SELECT COUNT(*) AS orders
FROM orders
WHERE store_id = 1
""").show()

## Exercise 2. Customers not referred by customer 24

How many customers were referred by someone other than customer 24? Count the customers nobody referred too. Before you run
anything, write one sentence in the first cell saying what the query will return and which customers you expect it to include.

In [ ]:
# My expected result, in words:
#

In [ ]:
# Your query: customers not referred by customer 24

## Exercise 3. Spot the flaw

A colleague wants to know how many customers were not referred by anyone, and thinks of a missing referrer as a 0. The query
runs without an error, but the result suggests that every customer was referred. Find the flaw, then write a query that works.

In [ ]:
con.sql("""
SELECT COUNT(*) AS not_referred
FROM customers
WHERE referred_by = 0
""").show()

In [ ]:
# Your corrected query: